**Step 1- Sample NER Dataset**

In [54]:
#[sentence, tags] pairs
data = [
    (
        ["Deepika", "works", "at", "Google", "in", "Hyderabad"],
        ["B-PER",   "O",      "O",  "B-ORG", "O",  "B-LOC"]
    ),
    (
        ["Elon",   "Musk", "founded", "Tesla", "in", "Califonia"],
        ["B-PER", "I-PER",   "O",     "B-ORG", "O",  "B-LOC"]
    ),
    (
        ["Microsoft", "is", "based", "in", "seattle"],
        ["B-ORG",     "O",  "O",     "O",  "B-LOC"]
    ),
    (
        ["Sundar", "Pichai", "leads", "Google", "from", "New", "York"],
        ["B-PER",  "I-PER", "O",     "B-ORG",  "O",    "B-LOC", "I-LOC"]
    ),
    (
        ["Priya", "studied", "at", "IIT",  "in", "Mumbai"],
        ["B-PER",    "O",     "O", "B-ORG", "O", "B-LOC"]
    ),
    (
        ["Subhashree", "Peddi", "eating", "dinner", "in", "South", "Korea"],
        ["B-PER",      "I-PER", "O",      "O",      "O",  "B-LOC", "I-LOC"]
    ),
]
        

**Step 2- Build Vocabulary**

In [55]:
#Word vocabulary
all_words =[]
for sentence, _ in data:
    all_words.extend(sentence)

word2idx = {"<PAD>": 0, "<UNK>":1}
for w in all_words:
    if w not in word2idx:
        word2idx[w] = len(word2idx)

# Tag Vocabulary
tag2idx = {
    "<PAD>"    : 0,
    "O"      : 1,
    "B-PER"  : 2,
    "I-PER"  : 3,
    "B-ORG"  : 4,
    "I-ORG"  : 5,
    "B-LOC"  : 6,
    "I-LOC"  : 7
}
idx2tag = {v: k for k, v in tag2idx.items()}

print("Vocab size:", len(word2idx))
print("Tag size:", len(tag2idx))

Vocab size: 33
Tag size: 8


**Step 3- Encode Dataset**

In [56]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 10

def encode_sentence(sentence, word2idx, max_len):
    encoded = [word2idx.get(w.lower(), word2idx["<UNK>"]) for w in sentence]

    #Pad
    encoded += [word2idx["<PAD>"]] * (max_len - len(encoded))
    return encoded[:max_len]

def encode_tag(tag, tag2idx, max_len):
    encoded = [tag2idx[t] for t in tag]

    #Pad with o tag
    encoded += [tag2idx["<PAD>"]] * (max_len - len(encoded))
    return encoded[:max_len]

class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx, max_len=MAX_LEN):
        self.x = [encode_sentence(s, word2idx, max_len) for s , _ in data]
        self.y = [encode_tag(t, tag2idx, max_len) for _ , t in data]

    def __len__(self):
            return len(self.x)
   
    def __getitem__(self, idx):
        return(
            torch.tensor(self.x[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long)
        )

dataset = NERDataset(data, word2idx, tag2idx)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print("Dataset size:", len(dataset))

Dataset size: 6


**Step 4 — LSTM NER Model**

In [57]:
import torch.nn as nn

class NERModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_tags):
        super(NERModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size,
                            hidden_size,
                            batch_first=True,
                            bidirectional=True) 
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * 2, num_tags)

    def forward(self,x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        output = self.dropout(output)
        logits = self.fc(output)

        return logits

model = NERModel(
    vocab_size = len(word2idx),
    embed_size = 16,
    hidden_size = 32,
    num_tags = len(tag2idx)
)

print(model)       

NERModel(
  (embedding): Embedding(33, 16, padding_idx=0)
  (lstm): LSTM(16, 32, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=8, bias=True)
)


**Step 5 — Train**

In [65]:
criterion = nn.CrossEntropyLoss(ignore_index=tag2idx["<PAD>"])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs=100

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()

        logits = model(x_batch)

        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            y_batch.reshape(-1)
        )
                         
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if(epoch + 1) % 10 == 0:
            print(f"Epoch{epoch+1:2d}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

Epoch10/100 | Loss: 0.5240
Epoch20/100 | Loss: 0.3662
Epoch30/100 | Loss: 0.3239
Epoch40/100 | Loss: 0.1927
Epoch50/100 | Loss: 0.1467
Epoch60/100 | Loss: 0.1108
Epoch70/100 | Loss: 0.0851
Epoch80/100 | Loss: 0.0656
Epoch90/100 | Loss: 0.0513
Epoch100/100 | Loss: 0.0384


**Step 6 — Predict Entities**

In [67]:
def predict_ner(sentence, model, word2idx, idx2tag, max_len=MAX_LEN):
    model.eval()

    tokens = sentence.split()
    tokens_lower = [w.lower() for w in tokens]
    
    encoded = encode_sentence(tokens_lower, word2idx, max_len)
    tensor = torch.tensor([encoded], dtype=torch.long)

    with torch.no_grad():
        logits = model(tensor)
        predictions = logits.argmax(dim=2)[0]

        print(f"\n{'word':<15} {'Predicted Tag'}")
        print("_" * 30)

        for i, token in enumerate(tokens):
            tag = idx2tag[predictions[i].item()]
            print(f"{token:<15} {tag}")

#_________________________Test1_________________________
predict_ner("Deepika works at Google in Hyderabad", model, word2idx, idx2tag)
predict_ner("Subhashree Peddi eating dinner in South Korea", model, word2idx, idx2tag)

# ── Test with new sentence! ───────────────────
predict_ner("Sundar Pichai leads Google in New York",
             model, word2idx, idx2tag)


word            Predicted Tag
______________________________
Deepika         B-PER
works           O
at              O
Google          B-ORG
in              O
Hyderabad       B-LOC

word            Predicted Tag
______________________________
Subhashree      B-PER
Peddi           I-PER
eating          O
dinner          O
in              O
South           B-LOC
Korea           I-LOC

word            Predicted Tag
______________________________
Sundar          B-PER
Pichai          I-PER
leads           O
Google          B-ORG
in              O
New             B-LOC
York            I-LOC
